# Course 27666 AI-guided Protein Science

# Graph Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_graph_machine_learning/blob/main/notebooks/shallow_embeddings.ipynb)

## Environment Setup

To perform deep learning on graph structures, we rely on PyTorch Geometric (PyG), a specialized extension library built on top of PyTorch. PyG leverages highly optimized specialized C++/CUDA extensions (`torch-cluster`, `pyg-lib`, etc.) to run sparse graph operations like message passing and random walks efficiently. Since these wheels are tightly coupled with specific compiled versions of PyTorch and CUDA, we query the exact system specifications dynamically to resolve and pull down the correct binary packages from PyG's hosted index.

In [ ]:
# Add this in a Google Colab cell to install the correct version of Pytorch Geometric.
import torch

def format_pytorch_version(version):
  return version.split('+')[0]

TORCH_version = torch.__version__
TORCH = format_pytorch_version(TORCH_version)

def format_cuda_version(version):
  return 'cu' + version.replace('.', '')

CUDA_version = torch.version.cuda
CUDA = format_cuda_version(CUDA_version)

!pip install torch-cluster -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric
!pip install pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

## Fetching Protein Annotation Stream from UniProt KB

We establish our core environment. We import `pandas` and `numpy` for classical structural array and tabular operations, `torch` for graph operations, and `urllib` to handle networking with remote programmatic databases.

The UniProt Knowledgebase (UniProtKB) provides functional protein annotations. Here, we build a query focused on verified human proteins (organism_id:9606, reviewed:true) restricted to balanced sequence sizes (length:[80 TO 500]) to request a stream of tab-separated values containing protein accessions matched with their curated subcellular spatial location annotations.

In [ ]:
import pandas as pd
import numpy as np
import urllib.parse

# UniProt data collection
organism = 9606
fields = ",".join(['accession','cc_subcellular_location'])
query = f'((organism_id:{organism}) AND (reviewed:true) AND (length:[80 TO 500]))'

url = 'https://rest.uniprot.org/uniprotkb/stream?' + urllib.parse.urlencode({
    'compressed':'false',
    'fields':fields,
    'format':'tsv',
    'query':query
})

df = pd.read_csv(url, sep='\t')
df = df.dropna()

## Location Labels

Curated subcellular information is delivered as unstructured text. To convert this text into labels for machine learning, we process the text to extract cytoplasmic/cytosolic or membrane annotations.

In [ ]:

def get_location_label(data):
    data = str(data).lower()
    if ((('cytosol' in data) or ('cytoplasm' in data)) and ('membrane' not in data)):
        return 'cytosolic'
    elif ('membrane' in data) and not (('cytosol' in data) or ('cytoplasm' in data)):
        return 'membrane'
    return np.nan

df['label'] = df['Subcellular location [CC]'].apply(get_location_label)
df = df.dropna(subset=['label'])

df.head()

,Entry,Subcellular location [CC],label
0,A0A096LP01,SUBCELLULAR LOCATION: Mitochondrion outer memb...,membrane
1,A0A0K2S4Q6,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...,membrane
2,A0A2R8Y7D0,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...,cytosolic
4,A0AVI4,SUBCELLULAR LOCATION: Endoplasmic reticulum me...,membrane
6,A0M8Q6,SUBCELLULAR LOCATION: Secreted {ECO:0000303|Pu...,membrane


## Download STRING interactions

The [STRING Database](https://string-db.org) captures physical and functional interactions among proteins. Each row indicates an edge between two proteins accompanied by a probabilistic combined_score scaled between 1 and 1000. To minimize background noise, we filter out weak records and retain only high-confidence interactions with scores $\ge 700$.

In [ ]:
string_url = "https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz"

ppi = pd.read_csv(
    string_url,
    sep=' '
)

ppi = ppi[ppi['combined_score'] >= 700]
ppi.head()

,protein1,protein2,combined_score
85,9606.ENSP00000000233,9606.ENSP00000158762,825
130,9606.ENSP00000000233,9606.ENSP00000357048,718
160,9606.ENSP00000000233,9606.ENSP00000262305,952
197,9606.ENSP00000000233,9606.ENSP00000329419,752
268,9606.ENSP00000000233,9606.ENSP00000469035,795


## Resolving Cross-Database Entry Identifiers (ID Mapping)

UniProt uses accessions (e.g., P26437) and STRING tracks proteins using Ensembl Protein identifiers (e.g., 9606.ENSP00000000233). To map from one identifier to another, we download STRING's alias mapping table, isolate valid UniProt_AC matches, and build a mapping dictionary.

In [ ]:
string_url = "https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz"

aliases = pd.read_csv(
    string_url,
    sep='\t'
)

aliases = aliases[
    aliases['source'].str.contains(r'\bUniProt_AC\b', na=False)
]

mapping = (
    aliases[['#string_protein_id', 'alias']]
    .drop_duplicates()
    .set_index('#string_protein_id')['alias']
    .to_dict()
)

ppi['protein1'] = ppi['protein1'].apply(lambda x: x if x not in mapping else mapping[x])
ppi['protein2'] = ppi['protein2'].apply(lambda x: x if x not in mapping else mapping[x])

In [ ]:
# Keep only proteins with labels and present in STRING
labeled = set(df['Entry'])
labeled_ppi = ppi[ppi['protein1'].isin(labeled) & ppi['protein2'].isin(labeled)]
labeled_ppi.head()

,protein1,protein2,combined_score
16980,O60359,Q2WGJ8,720
17029,O60359,Q7RTX7,723
17185,O60359,Q6PI25,835
17210,O60359,A8MZ26,720
17393,O60359,Q6ZSJ9,720


## Compiling the PyTorch Geometric Network Graph

To process graph data with PyTorch Geometric, we convert our interaction tables into a formal graph. We map every unique protein present in our filtered STRING subset to a continuous integer index spanning $[0, N-1]$. This mapping allows us to convert the string identifiers in our edge list into a numeric tensor of shape [2, E], representing the graph's structured Edge Index.

In [ ]:
from torch_geometric.data import Data
import torch

# Extract all unique proteins present in our filtered interaction subset
unique_proteins = pd.unique(ppi[['protein1', 'protein2']].values.ravel())
protein_to_idx = {protein: idx for idx, protein in enumerate(unique_proteins)}

edge_index = torch.tensor([
    [protein_to_idx[a] for a in ppi.protein1],
    [protein_to_idx[b] for b in ppi.protein2]
], dtype=torch.long)

graph = Data(edge_index=edge_index)

print(f"Compiled Graph Topology:")
print(f"Total Graph Nodes (Proteins): {len(unique_proteins)}")
print(f"Total Graph Edges (Interactions): {edge_index.shape[1]}")

Data(edge_index=[2, 473860])

## Instantiating and Optimizing Node2Vec Embeddings

The Node2Vec algorithm maps complex graph topologies into continuous, low-dimensional vector representations. It works by running biased random walks across the graph using two hyperparameters: the Return parameter ($p$), which controls how likely the walk is to immediately revisit a node, and the In-out parameter ($q$), which dictates whether the walk focuses locally (Breadth-First Search) or explores further outward (Depth-First Search).

The generated walk sequences are treated like sentences and fed into a skip-gram architecture ([Mikolov et al](https://arxiv.org/pdf/1301.3781)), which optimizes the representations so that proteins with similar interaction patterns end up close to each other in the vector space.

In [ ]:
from torch_geometric.nn import Node2Vec

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Instantiate Node2Vec model architecture
node2vec_model = Node2Vec(
    edge_index=edge_index,
    embedding_dim=128,      # Vector size per protein node
    walk_length=20,         # Length of random walks
    context_size=10,        # Context window size for skip-gram
    walks_per_node=10,      # Number of random walks started per node
    p=1.0,                  # Return parameter (higher encourages exploration)
    q=1.0,                  # In-out parameter (lower encourages local clustering)
    sparse=True             # Optimized sparse gradient tracking
).to(device)

# Initialize optimized sparse optimizer targeting lookup tables
loader = node2vec_model.loader(batch_size=1024, shuffle=True, num_workers=0)
optimizer = torch.optim.SparseAdam(list(node2vec_model.parameters()), lr=0.01)

# Training loop
node2vec_model.train()
for epoch in range(1, 21):  # Small iteration scope for fast verification
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = node2vec_model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch:02d} | Avg Node2Vec Unsupervised Loss: {total_loss / len(loader):.4f}")

0 9.096458554267883
1 7.448158949613571
2 6.04742556810379
3 4.88126277923584
4 3.9563260823488235
5 3.2714740335941315
6 2.77366042137146
7 2.4019026309251785
8 2.111798495054245
9 1.8817167654633522
10 1.6998507007956505
11 1.5553546398878098
12 1.4415270015597343
13 1.3521370440721512
14 1.2817538753151894
15 1.2267371788620949
16 1.185011386871338
17 1.1506058722734451
18 1.122503086924553
19 1.1007710471749306


## Supervised Classification Evaluation

Once the unsupervised Node2Vec training is done, we extract the learned embedding vectors for our labeled proteins. 

These graph-derived embeddings serve as structured feature matrices ($X$), while the cellular localization entries act as our targets ($y$). We then split the data into independent training and testing sets, and train a supervised logistic regression classifier to evaluate how effectively the structural graph properties predict localized biological functions.

In [ ]:
node2vec_model.eval()
embeddings = node2vec_model.embedding.weight
embedding_df = pd.DataFrame(
    embeddings,
    index=unique_proteins
)
embedding_df.head()


,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
9606.ENSP00000250825,-0.315393,-0.056171,-0.028523,0.210675,-0.102855,0.274423,0.713134,0.205866,-0.255967,0.555126,...,0.234977,0.082734,0.263093,-0.112709,-0.228498,-0.014479,0.233700,-0.682913,0.057860,-0.582904
9606.ENSP00000287844,0.022687,-0.008274,0.115236,-0.053235,-0.261566,0.044612,-0.045953,0.001823,-0.195384,0.273879,...,0.012830,-0.204738,-0.378405,0.107408,0.030180,0.099432,0.104148,-0.403887,0.294094,0.196109
9606.ENSP00000299157,-0.029667,0.026079,0.418375,0.248795,0.158859,0.137576,-1.111679,-0.092843,-0.140655,-0.271836,...,0.359997,-0.227600,0.044409,0.038801,-0.390493,-0.307936,-0.373165,0.089918,0.021835,-0.313675
9606.ENSP00000303178,0.103874,0.115293,0.003908,0.169141,-0.284767,0.437785,0.334375,-0.112949,-0.259753,0.131914,...,0.166046,0.221747,0.552759,0.200935,-0.258694,-0.318116,-0.072402,-0.372566,0.108094,-0.208651
9606.ENSP00000303300,0.332582,0.137934,0.020848,0.261754,-0.037439,-0.209561,0.599765,0.679780,-0.477038,-0.273706,...,0.625207,0.014097,0.274004,-0.123036,-0.672054,-0.406156,-0.425480,-0.355938,-0.250578,-0.130808


### Train localization classifier

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Compile the feature matrix by looking up the embeddings for our labeled proteins
train_rows = []
for _, row in df.iterrows():
    protein = row['Entry']
    if protein in embedding_df.index:
        train_rows.append(
            (
                embedding_df.loc[protein].values,
                row['label']
            )
        )
X = np.vstack([r[0] for r in train_rows])
y = np.array([r[1] for r in train_rows])

# Split the dataset into stratified training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train a classical downstream logistic regression classifier
clf = LogisticRegression(max_iter=5000)
clf.fit(X_train, y_train)


# Evaluate classification performance
pred = clf.predict(X_test)
print("\n--- Final Localization Prediction Performance Report ---")
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

   cytosolic       0.76      0.73      0.75       162
    membrane       0.79      0.81      0.80       199

    accuracy                           0.78       361
   macro avg       0.77      0.77      0.77       361
weighted avg       0.78      0.78      0.78       361

